#Messages
1. Human Message 
2. AI Message
3. System Message
4. Tool Message


{Role,
Content,
Metadata}

In [ ]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


from langchain.chat_models import init_chat_model
model=init_chat_model("groq:openai/gpt-oss-20b")
response=model.invoke("What is the capital of France?")
response.content

{'input_tokens': 78,
 'output_tokens': 39,
 'total_tokens': 117,
 'output_token_details': {'reasoning': 21}}

### Human Message  (Text Prompt)

In [7]:
response=model.invoke("What is the capital of France?")
response.content

'The capital of France is **Paris**.'

### List of Messages

In [15]:
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool


@tool
def get_ai_tutorials(topic: str) -> str:
    """Get a recommended learning path for an AI topic.
    Supported topics: langchain, langgraph, machine learning."""
    tutorials = {
        "langchain": "1) Models  2) Messages  3) Tools  4) Agents",
        "langgraph": "1) State  2) Nodes  3) Edges  4) Memory",
        "machine learning": "1) Python  2) NumPy/Pandas  3) Regression  4) Neural networks",
    }
    return tutorials.get(topic.lower(), f"No tutorial found for '{topic}'")


messages = [
    SystemMessage(content="You are an AI tutor. Use the get_ai_tutorials tool to give learning paths."),
    HumanMessage(content="I want to learn LangChain. What should I study first?"),
]

model_with_tools = model.bind_tools([get_ai_tutorials])

# Step 1: model decides to call the tool
ai_message = model_with_tools.invoke(messages)
print("Tool calls:", ai_message.tool_calls)
messages.append(ai_message)

# Step 2: run the tool and send the result back
for call in ai_message.tool_calls:
    result = get_ai_tutorials.invoke(call["args"])
    print("Tool result:", result)
    messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

# Step 3: model gives the final answer
final = model_with_tools.invoke(messages)
print("Final answer:\n", final.content)


Tool calls: [{'name': 'get_ai_tutorials', 'args': {'topic': 'langchain'}, 'id': 'fc_9cb48295-c288-4bca-9c05-b399ef2579db', 'type': 'tool_call'}]
Tool result: 1) Models  2) Messages  3) Tools  4) Agents
Final answer:
 **Getting started with LangChain – a quick “first‑step” guide**

Below is a concise learning path that the tool recommends, plus a brief explanation of why each step matters and some resources to dive into.

| # | Topic | Why it’s first | Quick‑start resources |
|---|-------|----------------|------------------------|
| 1 | **Models** | LangChain is all about orchestrating language‑model calls.  Understanding how to load, configure, and query a model (OpenAI, Anthropic, Cohere, etc.) is the foundation. | • [LangChain Models docs](https://python.langchain.com/docs/modules/models/) <br>• Intro video: *“LangChain for Beginners – Models Overview”* |
| 2 | **Messages** | Once you can talk to a model, you need to structure the conversation.  Messages (system, user, assistant) are

### Converstation History

In [18]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
system_message = SystemMessage(content="You are a helpful assistant.")
human_message = HumanMessage(content="I want to perform some calculations.")
ai_message = AIMessage(content="Sure! What calculations would you like to perform?")
human_message2 = HumanMessage(content="I want to calculate the sum of 5 and 10.")
messages = [system_message, human_message, ai_message, human_message2]

response = model.invoke(messages)
response.usage_metadata

{'input_tokens': 120,
 'output_tokens': 35,
 'total_tokens': 155,
 'output_token_details': {'reasoning': 13}}